# L3: Pipeline de vendas 

## Initial Imports

In [58]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

## Load API tokens for our 3rd party APIs

In [59]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

## Loading Tasks and Agents YAML files

In [60]:
# Define file paths for YAML configurations
files = {
    'lead_agents': 'config/lead_qualification_agents.yaml',
    'lead_tasks': 'config/lead_qualification_tasks.yaml',
    'email_agents': 'config/email_engagement_agents.yaml',
    'email_tasks': 'config/email_engagement_tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
lead_agents_config = configs['lead_agents']
lead_tasks_config = configs['lead_tasks']
email_agents_config = configs['email_agents']
email_tasks_config = configs['email_tasks']

## Create Pydantic Models for Structured Output

In [61]:
from pydantic import BaseModel, Field
from typing import Optional, List

class LeadPersonalInfo(BaseModel):
    name: str = Field(..., description="O nome completo do lead.")
    job_title: str = Field(..., description="O cargo do lead.")
    role_relevance: int = Field(..., ge=0, le=10, description="Uma pontuação que representa quão relevante é o papel do lead no processo de tomada de decisão (0-10).")
    professional_background: Optional[str] = Field(..., description="Uma breve descrição da experiência profissional do lead.")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="O nome da empresa onde o lead trabalha.")
    industry: str = Field(..., description="A indústria na qual a empresa atua.")
    company_size: int = Field(..., description="O tamanho da empresa em termos de número de funcionários.")
    revenue: Optional[float] = Field(None, description="A receita anual da empresa, se disponível.")
    market_presence: int = Field(..., ge=0, le=10, description="Uma pontuação que representa a presença de mercado da empresa (0-10).")

class LeadScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="A pontuação final atribuída ao lead (0-100).")
    scoring_criteria: List[str] = Field(..., description="Os critérios utilizados para determinar a pontuação do lead.")
    validation_notes: Optional[str] = Field(None, description="Quaisquer observações relacionadas à validação da pontuação do lead.")

class LeadScoringResult(BaseModel):
    personal_info: LeadPersonalInfo = Field(..., description="Informações pessoais sobre o lead.")
    company_info: CompanyInfo = Field(..., description="Informações sobre a empresa do lead.")
    lead_score: LeadScore = Field(..., description="A pontuação calculada e informações relacionadas ao lead.")


## Importing Tools

In [62]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

## Lead Qualification Crew, Agents and Tasks

In [63]:
# type: ignore
# Creating Agents
lead_data_agent = Agent(
  config=lead_agents_config['lead_data_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

cultural_fit_agent = Agent(
  config=lead_agents_config['cultural_fit_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

scoring_validation_agent = Agent(
  config=lead_agents_config['scoring_validation_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

# Creating Tasks
lead_data_task = Task(
  config=lead_tasks_config['lead_data_collection'],
  agent=lead_data_agent
)

cultural_fit_task = Task(
  config=lead_tasks_config['cultural_fit_analysis'],
  agent=cultural_fit_agent
)

scoring_validation_task = Task(
  config=lead_tasks_config['lead_scoring_and_validation'],
  agent=scoring_validation_agent,
  context=[lead_data_task, cultural_fit_task],
  output_pydantic=LeadScoringResult
)

# Creating Crew
lead_scoring_crew = Crew(
  agents=[
    lead_data_agent,
    cultural_fit_agent,
    scoring_validation_agent
  ],
  tasks=[
    lead_data_task,
    cultural_fit_task,
    scoring_validation_task
  ],
  verbose=True
)

2025-01-27 17:21:29,465 - 2952 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


LLM value is None
LLM value is None
LLM value is None


## Email Engagement Crew

In [64]:
# type: ignore
# Creating Agents
email_content_specialist = Agent(
  config=email_agents_config['email_content_specialist']
)

engagement_strategist = Agent(
  config=email_agents_config['engagement_strategist']
)

# Creating Tasks
email_drafting = Task(
  config=email_tasks_config['email_drafting'],
  agent=email_content_specialist
)

engagement_optimization = Task(
  config=email_tasks_config['engagement_optimization'],
  agent=engagement_strategist
)

# Creating Crew
email_writing_crew = Crew(
  agents=[
    email_content_specialist,
    engagement_strategist
  ],
  tasks=[
    email_drafting,
    engagement_optimization
  ],
  verbose=True
)

2025-01-27 17:21:29,484 - 2952 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


LLM value is None
LLM value is None


## Creating Complete Sales Flow

In [65]:
from crewai import Flow
from crewai.flow.flow import listen, start

class SalesPipeline(Flow):
    @start()
    def fetch_leads(self):
        # Puxar nossos leads do banco de dados
        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Diretor de Engenharia",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Usando o Agente de IA para melhorar o enriquecimento de dados."
                },
            },
        ]
        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)
        self.state["score_crews_results"] = scores
        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Aqui armazenaríamos as pontuações no banco de dados
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        return [score for score in scores if score['lead_score'].score > 70]

    @listen(filter_leads)
    def write_email(self, leads):
        scored_leads = [lead.to_dict() for lead in leads]
        emails = email_writing_crew.kickoff_for_each(scored_leads)
        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Aqui enviaríamos os emails para os leads
        return emails

flow = SalesPipeline()

## Plotting the Flow

In [66]:
flow.plot()

Plot saved as crewai_flow.html


In [67]:
import webbrowser
webbrowser.open('crewai_flow.html')

True

## Flow Kickoff

In [ ]:
emails = await flow.kickoff_async()

2025-01-27 17:21:29,630 - 2952 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


LLM value is already an LLM object
LLM value is already an LLM object
LLM value is already an LLM object
# Agent: Especialista em Dados de Leads
## Task: Coletar e analisar as seguintes informaÃ§Ãµes sobre o lead:
- InformaÃ§Ãµes Pessoais:
  - Nome: Obtenha o nome completo do lead.
  - Cargo: Determine o cargo atual do lead.
  - RelevÃ¢ncia do Cargo: Avalie quÃ£o relevante Ã© o cargo do lead para o processo de tomada de decisÃµes em uma escala de 0 a 10.
  - HistÃ³rico Profissional: Opcionalmente, colete uma breve descriÃ§Ã£o do histÃ³rico profissional do lead.

- InformaÃ§Ãµes da Empresa:
  - Nome da Empresa: Identifique o nome da empresa para a qual o lead trabalha.
  - IndÃºstria: Determine o setor em que a empresa opera.
  - Tamanho da Empresa: Estime o tamanho da empresa em termos de nÃºmero de funcionÃ¡rios.
  - Receita: Se disponÃ­vel, colete informaÃ§Ãµes sobre a receita anual da empresa.
  - PresenÃ§a no Mercado: Avalie a presenÃ§a da empresa no mercado em uma escala de 0 a 10

2025-01-27 17:21:32,771 - 2952 - serper_dev_tool.py-serper_dev_tool:170 - ERROR: Error making request to Serper API: 403 Client Error: Forbidden for url: https://google.serper.dev/search
Response content: b'{"message":"Unauthorized.","statusCode":403}'
2025-01-27 17:21:33,133 - 2952 - serper_dev_tool.py-serper_dev_tool:170 - ERROR: Error making request to Serper API: 403 Client Error: Forbidden for url: https://google.serper.dev/search
Response content: b'{"message":"Unauthorized.","statusCode":403}'
2025-01-27 17:21:33,468 - 2952 - serper_dev_tool.py-serper_dev_tool:170 - ERROR: Error making request to Serper API: 403 Client Error: Forbidden for url: https://google.serper.dev/search
Response content: b'{"message":"Unauthorized.","statusCode":403}'
2025-01-27 17:21:33,827 - 2952 - serper_dev_tool.py-serper_dev_tool:170 - ERROR: Error making request to Serper API: 403 Client Error: Forbidden for url: https://google.serper.dev/search
Response content: b'{"message":"Unauthorized.","status